# Notebook 1: GPIO Basics & Your First LED

Welcome to the first hands-on notebook in the AI Car workshop. Before this
robot can drive itself around, you need to understand the most basic building
block it's made of: **GPIO** (General Purpose Input/Output) pins, and how to
control one from Python.

In this notebook you will:

1. Learn what GPIO is, and the difference between an *input* and an *output*.
2. Learn about BCM pin numbering vs. physical pin numbering (a very common
   source of confusion for beginners).
3. Learn why Raspberry Pi GPIO is a **3.3V** system, and why that matters.
4. Wire up a real LED to your Raspberry Pi.
5. Control that LED from Python using the `gpiozero` library: turning it on,
   off, toggling it, and blinking it.
6. See how this project wraps that raw `gpiozero` code into a small reusable
   module (`src/hardware/led.py`), which is what later notebooks will
   actually use.
7. Practice with a few short exercises.

This is a low-risk notebook — a single LED and a resistor, nothing that can
be damaged by a mistake in code. That makes it the right place to build good
habits before later notebooks introduce motors and sensors that are less
forgiving.

## What is GPIO?

**GPIO** stands for **General Purpose Input/Output**. It refers to the rows
of metal pins along the edge of the Raspberry Pi's board. Each pin can be
told, in software, to behave as either:

- an **output** — the Pi actively drives the pin to a HIGH voltage (~3.3V) or
  LOW voltage (0V/ground), and something you've wired to that pin (like an
  LED) reacts to that voltage; or
- an **input** — the Pi *reads* whatever voltage is currently present on the
  pin (for example, from a button or a sensor) without driving it itself.

In this notebook, GPIO17 will be configured as an **output**: we are going to
command it HIGH or LOW, and an LED wired to it will light up or go dark in
response. Later notebooks (button, ultrasonic sensor) will use GPIO pins
configured as **inputs**, reading a signal instead of producing one.

## Wire your LED first — before running any code

**Do this now, before you run any cell below.** Running the code cells with
nothing wired up is safe (nothing will break), but you won't see anything
happen, and the point of this notebook is to *see* your code affect the real
world.

### What you need

- 1 LED (any color)
- 1 resistor, approximately **220-330 ohms** (this limits current through the
  LED so it doesn't burn out — never wire an LED directly across a supply
  with no resistor)
- 2 jumper wires (or wire the resistor/LED directly into the breadboard rails)
- A breadboard is easiest, but not required

### Where to connect

| Connect | To |
|---|---|
| GPIO17 (BCM numbering) | **Physical pin 11** on the 40-pin header |
| A GND (ground) pin | **Physical pin 9** — conveniently right next to pin 11 |

Wiring path: **GPIO17 (physical pin 11) → resistor (220-330Ω) → LED anode
(long leg) → LED cathode (short leg) → GND (physical pin 9)**

The resistor can sit anywhere in that series path (between the GPIO pin and
the LED, as shown above, is the conventional choice) — what matters is that
current has to pass through it on its way from GPIO17 to GND.

### Polarity matters

An LED is **directional** — it only lights up when current flows through it
one way. Look at the two legs:

- The **longer leg** is the **anode** (+) — this goes toward GPIO17 (through
  the resistor).
- The **shorter leg** is the **cathode** (-) — this goes toward GND.

If you also look closely at the LED body, the cathode side usually has a
small flat edge on the plastic rim, which is a backup way to identify it if
the legs have been trimmed to the same length.

If you get the polarity backwards, the LED simply won't light — it won't
damage anything at these voltages/currents. That said, wiring it correctly
the first time saves you a debugging step.

### Before you run anything

Double-check, by eye, that:

- [ ] The resistor is in the path between GPIO17 and the LED (or between the
      LED and GND — either side works, but *a* resistor must be present)
- [ ] The LED's long leg (anode) points toward GPIO17/the resistor
- [ ] The LED's short leg (cathode) points toward GND
- [ ] You're using physical pin 11 for GPIO17 and physical pin 9 for GND —
      not some other pin

**If the LED doesn't light up later in this notebook**, the code is almost
certainly not the problem — first recheck (in this order): polarity, that
the resistor/LED/wires are making solid contact, and that you're on the
correct physical pins (11 and 9). It is very easy to be one pin off on the
header and end up "working" on a pin that isn't actually GPIO17.

## BCM numbering vs. physical pin numbering

This trips up almost everyone the first time, so it's worth being explicit
about it now.

The Raspberry Pi's 40-pin header has pins that can be referred to **two
different ways**:

- **Physical numbering**: just counting the pins on the header, 1 through 40,
  by physical position (row by row, left to right).
- **BCM numbering** (Broadcom numbering): the number the Broadcom SoC itself
  uses internally to address that pin, e.g. "GPIO17". This is the numbering
  scheme silkscreened on most pinout diagrams and the one this project uses
  everywhere in code (see `src/config.py` and `PINOUT.md` in the project
  root).

These two numbers are **not the same**, and there's no simple formula between
them — you look them up on a pinout diagram (or run `pinout` from a terminal
on the Pi, if the `gpiozero` package's command-line tool is installed).

For this notebook: **GPIO17 in BCM numbering is physical pin 11**. When you
write `LED(17)` in Python (as you will shortly), gpiozero interprets that
`17` as a BCM number, not a physical pin count — which is why the wiring
instructions above pointed you at physical pin 11, not "pin 17" on the
header.

This project standardizes on BCM numbering everywhere in code (see the
comment at the top of `src/config.py`), so once you've got this straight for
GPIO17, the same mental model applies to every pin used in later notebooks.

## 3.3V logic — and why this matters later

The Raspberry Pi's GPIO pins operate at **3.3 volts**, not 5 volts. "HIGH"
means approximately 3.3V, and "LOW" means 0V (ground). This is different from
some other hobbyist boards (like classic Arduino Unos) that use 5V logic.

**Important safety rule you'll need again later:** Raspberry Pi GPIO *input*
pins can be damaged by voltages higher than 3.3V. You must never connect a
5V signal directly into a Pi GPIO pin.

This isn't a concern for today's notebook — the LED is a passive output-side
component and physically cannot push voltage back into the Pi. But keep this
rule in mind, because it becomes directly relevant in the **ultrasonic
sensor notebook** later in this course: the HC-SR04 ultrasonic sensor's ECHO
pin outputs a 5V signal, and it must be stepped down with a voltage divider
before it's safe to connect to a Pi GPIO input. `src/config.py` and
`PINOUT.md` already have a note on this next to `ULTRASONIC_ECHO_PIN`, well
ahead of that notebook, for exactly this reason. For now, just file the rule
away: **3.3V only, into or out of a Pi GPIO pin.**

## GND (ground)

Every circuit needs a complete loop for current to flow around — a voltage
source (GPIO17, when driven HIGH) and a return path back to that source's
reference point, which is **ground (GND)**. The Raspberry Pi's 40-pin header
has several pins labeled GND (physical pins 6, 9, 14, 20, 25, 30, 34, and 39)
— they're all connected together internally, so any one of them works. We're
using physical pin 9 simply because it's right next to physical pin 11
(GPIO17), which makes for a short, tidy connection on the breadboard.

Without a GND connection, the circuit is incomplete and the LED will not
light, no matter what the code does — there's nowhere for the current to go.

## Importing gpiozero

`gpiozero` is a Python library that wraps the low-level details of talking
to GPIO pins in simple, friendly classes. Instead of manually setting pin
modes and voltage levels, you create an object representing the *device*
you've wired up (here, an `LED`) and call methods on it like `.on()` and
`.off()`.

### Explanation

Import the `LED` class from `gpiozero`. This doesn't touch any hardware yet
— it just makes the class available to use.

In [ ]:
from gpiozero import LED


### Expected result

No output. If there's no error, the import worked. (If you get an
`ImportError` or `ModuleNotFoundError`, `gpiozero` isn't installed in the
Jupyter kernel's Python environment — this shouldn't happen on the
project Pi, since it was confirmed installed system-wide during setup.)

### Physical result

Nothing — no hardware has been touched yet.

## Creating the LED object

### Explanation

This line creates an `LED` object bound to **BCM pin 17** — the pin you
wired your LED to. Creating this object is what actually reserves/configures
GPIO17 as an output pin behind the scenes; from this point on, this
notebook's Python process "owns" GPIO17 until you release it later with
`.close()` (or the notebook's kernel is restarted/shut down).

In [ ]:
led = LED(17)


### Expected result

No output (or, in some environments, an object repr like
`<gpiozero.LED object on pin GPIO17, active_high=True, is_active=False>`
if you inspect `led` in a cell by itself). No error.

If this raises a `GPIOZeroPinFactoryFallback` warning or a `BadPinFactory` /
permission-related error instead, it usually means either something else on
the Pi already has GPIO17 open (restart the kernel and try again) or the
user account isn't in the `gpio` group — shouldn't be the case here, since
`admin` is already confirmed to be in `gpio`.

### Physical result

The LED should currently be **off** (gpiozero defaults a newly-created `LED`
to the off state). Nothing should have visibly changed yet.

## Turning the LED on

### Explanation

`.on()` drives GPIO17 HIGH (~3.3V). Current now flows: GPIO17 → resistor →
LED anode → LED cathode → GND.

In [ ]:
led.on()


### Expected result

No output, no error — `.on()` returns `None`.

### Physical result

**The LED should light up now.** If it doesn't, stop here and recheck your
wiring (polarity first, then that pins 11 and 9 are the ones actually used)
before assuming anything is wrong with the code — this exact line is known
to work against `src/hardware/led.py`'s equivalent call, which QA already
verified toggles GPIO17's electrical level correctly.

## Turning the LED off

### Explanation

`.off()` drives GPIO17 LOW (0V). With no voltage difference across the
resistor/LED, current stops flowing and the LED goes dark.

In [ ]:
led.off()


### Expected result

No output, no error.

### Physical result

The LED should turn off.

## Toggling the LED

### Explanation

`.toggle()` flips the LED to whatever state it *isn't* currently in — off
becomes on, on becomes off. This is convenient when you don't want to track
the current state yourself. Run the cell below more than once and watch the
LED flip each time.

In [ ]:
led.toggle()


### Expected result

No output, no error, each time you run the cell.

### Physical result

Since the LED was off after the previous cell, this first `.toggle()` should
turn it **on**. Run the same cell again and it should turn back **off**.

## Blinking the LED with a loop

### Explanation

A "blink" is just alternating `.on()` and `.off()` with a pause in between,
using Python's built-in `time.sleep()` (which pauses execution for a given
number of seconds). The loop below blinks the LED 5 times, with the LED on
for 0.5 seconds and off for 0.5 seconds each cycle, printing progress as it
goes so you can follow along even without watching the LED.

In [ ]:
import time

for i in range(5):
    print(f"Blink {i + 1}/5: ON")
    led.on()
    time.sleep(0.5)

    print(f"Blink {i + 1}/5: OFF")
    led.off()
    time.sleep(0.5)

print("Done blinking.")


### Expected result

Ten `Blink N/5: ON`/`OFF` lines printed in order, one pair per second, ending
with `Done blinking.`. The cell will take about 5 seconds to finish running
(you'll see the `[*]` busy indicator next to the cell the whole time).

### Physical result

The LED should visibly blink on and off, twice per second, five times.

## Cleanup: releasing the GPIO pin

### Explanation

When you're done with a `gpiozero` device, it's good practice to call
`.close()` on it. This releases GPIO17 so it can be reused — by another
notebook cell creating a fresh `LED(17)`, by a different program, or simply
so the pin isn't left in a state some other code doesn't expect. If you skip
this and just keep going, gpiozero will still clean up automatically when the
kernel eventually shuts down or restarts — but it's better to be explicit,
especially once later notebooks are juggling several pins/devices at once
(motors, sensors) where leaving things open can cause confusing "pin already
in use" errors.

In [ ]:
led.close()


### Expected result

No output, no error.

### Physical result

The LED should be off (if it wasn't already), and GPIO17 is now released. If
you try to use the `led` variable again after this (e.g. `led.on()`), you'll
get an error, since the underlying pin has been closed — you'd need to
re-run `led = LED(17)` first.

## From raw `gpiozero` calls to a reusable project module

Everything above — `from gpiozero import LED`, `LED(17)`, `.on()`, `.off()`,
`.close()` — is perfectly good Python, and it's important to understand it at
that level first. But imagine writing that out, by hand, in every single
notebook and script in this project, every time any code needs to touch the
status LED. That gets repetitive fast, and if the LED's pin ever changed,
you'd have to hunt down and fix every copy.

This project solves that the standard way: by wrapping the raw `gpiozero`
calls in a small, reusable **module** — `src/hardware/led.py` — with a
single **named constant** for the pin number — `LED_PIN` in `src/config.py`.
From now on, any code in this project (including later notebooks) that wants
to control this LED imports from those two files instead of repeating the
raw `gpiozero` calls.

Take a look at what's in there:

- **`src/config.py`** defines `LED_PIN = 17` — one single source of truth
  for which BCM pin the LED is on. If the wiring ever moved to a different
  pin, this is the only line that would need to change.
- **`src/hardware/led.py`** defines small functions — `get_led(pin=LED_PIN)`,
  `led_on(led)`, `led_off(led)`, `toggle_led(led)`, `cleanup(led)` — that are
  thin wrappers around exactly the `gpiozero.LED` calls you just used by
  hand above. Each function does one job, which makes each one easy to
  teach, test, and reuse in its own right.

This is the same LED, the same GPIO17, the same underlying `gpiozero.LED`
object — just organized so the rest of the project (and you, in later
notebooks) doesn't have to keep rewriting it.

### Explanation

To import from `src/` inside a notebook that lives in `notebooks/`, we need
to add the `src/` folder to Python's module search path (`sys.path`) first —
notebooks aren't automatically able to see it otherwise. This only needs to
be done once per kernel session.

In [ ]:
import sys
sys.path.insert(0, '../src')

from hardware.led import get_led, led_on, led_off, toggle_led, cleanup
from config import LED_PIN

print(f"LED_PIN from config.py: {LED_PIN}")


### Expected result

`LED_PIN from config.py: 17` printed, no errors. If you get a
`ModuleNotFoundError: No module named 'hardware'` or `'config'`, double check
this notebook is running from the `notebooks/` folder so that `../src`
actually resolves to the project's `src/` directory.

### Physical result

Nothing yet — this cell only imports code, it doesn't touch the LED.

### Explanation

Now do the exact same on/off sequence as before, but through the project's
module functions instead of raw `gpiozero` calls. `get_led()` defaults to
`LED_PIN` (17) so we don't even need to pass a pin number — the module
already knows which pin from `config.py`.

In [ ]:
led = get_led()          # uses LED_PIN (17) from config.py by default
led_on(led)
time.sleep(1)
led_off(led)
cleanup(led)
print("Module-based LED sequence complete.")


### Expected result

`Module-based LED sequence complete.` printed after about a 1-second pause,
no errors.

### Physical result

The LED should turn on, stay on for about 1 second, then turn off — same
physical behavior as the raw `gpiozero` calls earlier, just driven through
`src/hardware/led.py` this time. This is exactly the pattern (`get_led()` →
some on/off/toggle calls → `cleanup()`) that later notebooks in this project
will reuse for the RGB LED, and that later hardware modules (motors,
ultrasonic) will follow the same shape for.

## Safety recap

- This notebook's circuit is low-risk: a single LED and a current-limiting
  resistor, both passive components that can't feed voltage back into the
  Pi or draw more current than the resistor allows.
- The one safety rule to carry forward from here: **never connect a GPIO pin
  directly to a 5V signal.** Raspberry Pi GPIO is a 3.3V-only system, and a
  5V input can damage the pin (or the whole board). This didn't come up
  today because the LED circuit never introduces 5V anywhere, but it will
  matter directly once you reach the **ultrasonic sensor notebook**, where
  the HC-SR04's ECHO pin outputs 5V and must be stepped down with a voltage
  divider before it ever reaches a GPIO input. See `SAFETY.md` in the
  project root for the full, growing list of safety rules as later notebooks
  add motors and sensors.

## Exercises

Try these on your own. There's no single "right" answer for most of these —
the goal is to get comfortable changing the code and predicting what should
happen physically before you run it.

**1. Change the blink timing.**
Copy the blink loop cell from earlier into a new cell below, and change the
`time.sleep(0.5)` values so the LED blinks twice as fast (0.25s on/off) or
twice as slow (1.0s on/off). Before running it, predict how long the whole
loop will take to finish for 5 blinks — then check by watching the "busy"
indicator on the cell.

**2. Move the LED to a different GPIO pin.**
Pick a different, currently-unused BCM pin (do *not* use any pin already
listed in `src/config.py` — those are reserved for other components).
Rewire your LED and resistor from physical pin 11 to the new pin's physical
location (look it up on a pinout diagram first!), then either pass the new
BCM number directly (`LED(<new_pin>)`), or — to practice the "project module"
pattern from this notebook — imagine editing `LED_PIN` in `src/config.py` to
point at the new pin (don't actually edit the file for this exercise; just
write out, in a markdown cell, what line you'd change). Why does the project
prefer changing one constant in `config.py` over changing `17` everywhere
it's hard-coded?

**3. Build a simple on/off pattern.**
Using `.on()`, `.off()`, and `time.sleep()`, write a cell that blinks out a
short recognizable rhythm instead of even blinks — for example Morse code
SOS (short-short-short, long-long-long, short-short-short: try 0.2s on for
"short" and 0.6s on for "long", with a 0.2s off gap between each), or any
rhythm of your own choosing (e.g. "on 0.1s, off 0.1s" three times fast,
then "on 1s, off 0.5s" twice slow).

**4. (Optional, extra challenge) Blink a countable number of times based on
input.**
Write a cell that reads an integer from the user (`n = int(input("How many
blinks? "))`) and blinks the LED exactly `n` times using a loop. This
previews a pattern (combining a loop with a runtime value) you'll reuse when
notebooks introduce sensor readings that vary each run.